# Baseline Evaluation

In [ ]:
import torch
from pathlib import Path
import pandas as pd
from transformers import AutoTokenizer
from html import escape
from IPython.display import HTML, display

## 1. Verify GPU access via collab (Tesla T4)

In [20]:
print("GPU available:", torch.cuda.is_available())
if torch.cuda.is_available():
    print("GPU:", torch.cuda.get_device_name(0))

GPU available: False


## 2. Load stratified baseline datasets


In [21]:
DATA_DIRECTORY = Path("/content/stratified-baseline")
LABEL_COLUMN = "label (target)"

train = pd.read_csv(DATA_DIRECTORY / "train.csv", dtype={"paper_id": "string"})
validation = pd.read_csv(DATA_DIRECTORY / "validation.csv", dtype={"paper_id": "string"})
test = pd.read_csv(DATA_DIRECTORY / "test.csv", dtype={"paper_id": "string"})

print("Training:", train.shape)
print("Validation:", validation.shape)
print("Test:", test.shape)
print("Sample records from the training set:")
print(train.head())

Training: (2000, 7)
Validation: (500, 7)
Test: (500, 7)
Sample records from the training set:
                                   paper_id  \
0  c655efe138c190df9fab6f7f8ce218566e660023   
1  5ab10f3cf7347f83ddce8e5a2aeb0585c1847764   
2  541aaa4302dfb75b5eca3809a80ddbc7b69aab41   
3  c52a320fca80026e22b4a7130d6210c7490cef23   
4  f1dd0cda5bb31ad2ab5f7dbc6b187f81159f1651   

                                               title  \
0  Dose correspondence between olanzapine long-ac...   
1  Sustained efficacy of adefovir add-on therapy ...   
2  Impaired coronary blood flow in nonculprit art...   
3  Characterization of apoptosis and autophagy th...   
4  Beta-blocker therapy for cardiac dysfunction i...   

                                            abstract  source_row quality_flag  \
0  Oral-to-depot dose correspondence was explored...        9446          NaN   
1  BACKGROUND\nCurrently, there is no consensus o...       10613          NaN   
2  OBJECTIVES AND BACKGROUND\nWhile attenti

## 3. Tokenize titles and abstracts

Use the standard English [BERT-base uncased checkpoint](https://huggingface.co/google-bert/bert-base-uncased).
The tokenizer converts each title and abstract into `[CLS] title [SEP] abstract [SEP]`
and prepares token IDs, attention masks, and segment IDs for BERT.

For this trial, pad or truncate each input to **256 tokens**, including special tokens.
Use the same settings for all splits. Tokenization does not train a model or use labels;
the test inputs remain reserved for final evaluation.


In [22]:
%pip install -q transformers

In [23]:
MODEL_NAME = "google-bert/bert-base-uncased"
MAX_LENGTH = 256

tokenizer = AutoTokenizer.from_pretrained(MODEL_NAME, use_fast=True)


def tokenize_papers(frame):
    return tokenizer(
        text=frame["title"].tolist(),
        text_pair=frame["abstract"].tolist(),
        padding="max_length",
        truncation=True,
        max_length=MAX_LENGTH,
        return_tensors="pt",
    )


train_inputs = tokenize_papers(train)
validation_inputs = tokenize_papers(validation)
test_inputs = tokenize_papers(test)

print("Checkpoint:", MODEL_NAME)
print("Training token IDs:", tuple(train_inputs["input_ids"].shape))
print("Validation token IDs:", tuple(validation_inputs["input_ids"].shape))
print("Test token IDs:", tuple(test_inputs["input_ids"].shape))


Checkpoint: google-bert/bert-base-uncased
Training token IDs: (2000, 256)
Validation token IDs: (500, 256)
Test token IDs: (500, 256)


### Tokenizer example: input and output

In [ ]:
EXAMPLE_INDEX = 0
example = train.iloc[EXAMPLE_INDEX]

input_ids = train_inputs["input_ids"][EXAMPLE_INDEX].tolist()
token_table = pd.DataFrame({
    "Position": range(len(input_ids)),
    "Token": tokenizer.convert_ids_to_tokens(input_ids),
    "Token ID": input_ids,
    "Attention": train_inputs["attention_mask"][EXAMPLE_INDEX].tolist(),
    "Segment": train_inputs["token_type_ids"][EXAMPLE_INDEX].tolist(),
})

input_html = (
    "<h4>Title</h4><p>" + escape(example["title"]) + "</p>"
    "<h4>Abstract</h4><p>" + escape(example["abstract"]) + "</p>"
)
output_html = token_table.to_html(index=False, border=0, escape=True)

display(HTML(f"""
<table style="width:100%; table-layout:fixed; text-align:left;">
  <thead>
    <tr>
      <th style="width:50%; text-align:left; padding:12px;">Original input</th>
      <th style="width:50%; text-align:left; padding:12px;">Tokenizer output ({len(input_ids)} positions)</th>
    </tr>
  </thead>
  <tbody>
    <tr>
      <td style="vertical-align:top; padding:12px;">
        <div style="max-height:500px; overflow:auto; white-space:normal; overflow-wrap:anywhere; text-align:left;">
          {input_html}
        </div>
      </td>
      <td style="vertical-align:top; padding:12px;">
        <div style="max-height:500px; overflow:auto; text-align:left;">
          {output_html}
        </div>
      </td>
    </tr>
  </tbody>
</table>
"""))


## 4. Pass tokens to BERT